In [1]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
            ELSE '中频搜索词'
         END AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df.head(2)

2025-04-07 18:40:58 - INFO - Thread count: 20
2025-04-07 18:41:30 - INFO - Tunnel session created: <InstanceDownloadSession id=20250407184130981b481a0c0ae01a project_name=summerfarm_ds instance_id=20250407104059421gz9inoze19g>
2025-04-07 18:41:32 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS s

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,芒果,11431,178549,56175,5.7,315.0,0.0,7.9,4.0,8.0,12.0,高频搜索词,0.314620,60,高点击率词
1,牛奶,11136,115817,36021,12.5,304.0,0.0,12.9,9.0,18.0,27.0,高频搜索词,0.311017,60,高点击率词


In [2]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    """
    从SLS(Simple Log Service)获取指定日期的用户变体数据。

    Args:
        day (datetime): 要获取数据的日期。
        check_if_local_exist (bool): 是否检查本地数据库中是否已存在数据，默认为True。

    Returns:
        pd.DataFrame: 包含用户变体数据的DataFrame。
    """
    # 构建数据库文件名和表名
    db_file_name = f"./data/search_ab_202504_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    # 连接到SQLite数据库
    conn = sqlite3.connect(db_file_name)

    # 如果设置为检查本地数据
    if check_if_local_exist:
        try:
            # 尝试从数据库中读取数据
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            # 关闭数据库连接
            conn.close()
            # 返回读取的数据
            return df
        except pd.io.sql.DatabaseError:
            # 如果表不存在，则忽略错误
            pass

    # 构建SLS查询语句
    query = f"""
type:a and (ap:/mall/sku/page or ap:"/product/6" ) and (pageName:/search/goods or pageName: "/search/goods-new")|
select array_join(array_sort(array_agg(distinct regexp_replace(ap, '\d+','{{digit}}'))),',') as api_list,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[0].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_sort(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[0].variantId'))),',') variant_list
from log group by 2,3,4,5,6 having experiment_id = 'search_new_ui202504' limit 1000000"""
    # 设置查询的起始时间和结束时间
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    # 从SLS获取数据
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",  # 指定SLS项目
        logstore="xm-mall",  # 指定SLS日志库
        from_time=from_time,  # 指定查询起始时间
        to_time=to_time,  # 指定查询结束时间
    )

    # 将search_times列中的缺失值填充为1，并转换为整数类型
    _df["search_times"] = _df["search_times"].fillna(1).astype(int)
    # 将variant_list列中的缺失值填充为"none"
    _df["variant_list"] = _df["variant_list"].fillna("none")

    # 如果DataFrame不为空
    if not _df.empty:
        # 删除不需要的列
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        # 将数据写入SQLite数据库，如果表已存在则替换
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    # 关闭数据库连接
    conn.close()
    # 返回数据
    return _df


# 创建一个空的DataFrame来存储所有日期的用户变体数据
all_user_variant_df = pd.DataFrame()
# 设置起始日期和结束日期
start_date = datetime(2025, 4, 3)
end_date = datetime.now()
# 从起始日期开始循环，直到结束日期
current_date = start_date
while current_date <= end_date:
    # 检查是否是今天
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    # 如果是今天,则跳过，因为今天的数据可能不完整
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    # 获取当前日期的用户变体数据
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    # 将当前日期的数据添加到总的DataFrame中
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    # 日期增加一天
    current_date += timedelta(days=1)

# 显示前10行数据
all_user_variant_df.head(10)

即将获取数据: =====> 2025-04-05 00:00:00 2025-04-05 23:59:59.999999 xm-mall: 
type:a and (ap:/mall/sku/page or ap:"/product/6" ) and (pageName:/search/goods or pageName: "/searc
>=====数条数:7874
即将获取数据: =====> 2025-04-06 00:00:00 2025-04-06 23:59:59.999999 xm-mall: 
type:a and (ap:/mall/sku/page or ap:"/product/6" ) and (pageName:/search/goods or pageName: "/searc
>=====数条数:7536
今天的数据还未完整，跳过:2025-04-07 00:00:00


,api_list,page_ame,experiment_id,type,uid,ds,search_times,variant_list
0,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,474174,20250403,18,V2
1,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,431483,20250403,7,V2
2,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,432031,20250403,11,V2
3,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,504394,20250403,1,V1
4,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,150920,20250403,2,V2
5,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,211761,20250403,2,V1
6,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,356827,20250403,12,V1
7,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,344928,20250403,3,V2
8,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,409045,20250403,15,V2
9,/product/{digit}/{digit},/search/goods,search_new_ui202504,a,419255,20250403,2,V1


In [3]:
import pandasql
stats=pandasql.sqldf("""select ds,variant_list,count(distinct uid) unique_user 
                     from all_user_variant_df group by ds,variant_list order by ds desc,variant_list""")

display(stats)

,ds,variant_list,unique_user
0,20250406,V1,4156
1,20250406,V2,1120
2,20250406,V3,1135
3,20250406,V4,1124
4,20250405,V1,2950
5,20250405,"V1,V2",95
6,20250405,V2,2935
7,20250405,V3,924
8,20250405,"V3,V4",40
9,20250405,V4,930


In [4]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and (pageName:/search/goods-new or pageName:/search/goods) |
select  pageName,regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,pageName,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_202504_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

即将获取数据: =====> 2025-04-05 00:00:00 2025-04-05 23:59:59.999999 xm-mall: 
type:view and (pageName:/search/goods-new or pageName:/search/goods) |
select  pageName,regexp_extr
>=====数条数:266826
即将获取数据: =====> 2025-04-06 00:00:00 2025-04-06 23:59:59.999999 xm-mall: 
type:view and (pageName:/search/goods-new or pageName:/search/goods) |
select  pageName,regexp_extr
>=====数条数:255306
今天的数据还未完整，跳过:2025-04-07 00:00:00


In [5]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and (pageName:/search/goods-new or pageName:/search/goods) |
select pageName,coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,pageName,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_202504_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2025-04-05 00:00:00, to_time:2025-04-05 23:59:59.999999, logstore:xm-mall, query:
type:cl and (pageName:/search/goods-new or pageName:/search/goods) |
select pageName,coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
 
>=====数据条数:33720
即将获取数据: =====>from_time:2025-04-06 00:00:00, to_time:2025-04-06 23:59:59.999999, logstore:xm-mall, query:
type:cl and (pageName:/search/goods-new or pageName:/search/goods) |
select pageName,coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
 
>=====数据条数:30597
今天的数据还未完整，跳过:2025-04-07 00:00:00


,pageName,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,/search/goods,null,阶梯价步进器,N001S01R005,加购弹窗,null,undefined,20250403,安佳淡奶油,cl,513815,/search/goods,undefined,"name:searchGoods,pdName:安佳淡奶油,type:2"
1,/search/goods,null,安佳淡奶油 1L*12盒,N001S01R005,加购弹窗,56,"name:安佳淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:337",20250403,安佳淡奶油,cl,513815,/search/goods,"name:安佳淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:337","name:searchGoods,pdName:安佳淡奶油,type:2"
2,/search/goods-new,null,台创蓝黛可可粉（低脂10%-12%）,642483031185,加购弹窗,7826,"name:台创蓝黛可可粉（低脂10%-12%）,pid:加购弹窗,sku:642483031185,pdid:7826,stock:0",20250403,可可粉,cl,17325,/search/goods-new,"name:台创蓝黛可可粉（低脂10%-12%）,pid:加购弹窗,sku:642483031185,pdid:7826,stock:0","name:Search,word:PET杯新品首发！,linkShadingWord:[object Object],isTiming:no"
3,/search/goods,null,步进器,661210374866,加购弹窗,null,undefined,20250403,C味三色大芋圆,cl,359593,/search/goods,undefined,"name:searchGoods,pdName:C味三色大芋圆,type:2"
4,/search/goods,null,步进器,661210374866,加购弹窗,null,undefined,20250403,C味三色大芋圆,cl,359593,/search/goods,undefined,"name:searchGoods,pdName:C味三色大芋圆,type:2"
5,/search/goods,null,步进器,661210374866,加购弹窗,null,undefined,20250403,C味三色大芋圆,cl,359593,/search/goods,undefined,"name:searchGoods,pdName:C味三色大芋圆,type:2"
6,/search/goods,null,加入购物车,661210374866,加购弹窗,3092,"name:加入购物车,pid:加购弹窗,sku:661210374866,pdid:3092,stock:47",20250403,C味三色大芋圆,cl,359593,/search/goods,"name:加入购物车,pid:加购弹窗,sku:661210374866,pdid:3092,stock:47","name:searchGoods,pdName:C味三色大芋圆,type:2"
7,/search/goods,8,云南蓝莓 125G*12盒/一级/果径12-14mm,567351400105,goods,12224,"idx:8,name:云南蓝莓 125G*12盒/一级/果径12-14mm,pid:goods,sku:567351400105,salePrice:95,pdid:12224,stock:1...",20250403,蓝莓,cl,36708,/search/goods,"idx:8,name:云南蓝莓 125G*12盒/一级/果径12-14mm,pid:goods,sku:567351400105,salePrice:95,pdid:12224,stock:1...","name:searchGoods,pdName:蓝莓,type:2"
8,/search/goods,1,新日清475Q大米预拌粉,674524418543,唤起购买,8028,undefined,20250403,大米,cl,356543,/search/goods,undefined,"name:searchGoods,pdName:大米,type:2"
9,/search/goods-new,0,朱师傅防潮可可粉,642437244575,goods,7561,"idx:0,name:朱师傅防潮可可粉,pid:goods,sku:642437244575,salePrice:59,pdid:7561,stock:10000,ext:cross",20250403,防潮可可粉,cl,17325,/search/goods-new,"idx:0,name:朱师傅防潮可可粉,pid:goods,sku:642437244575,salePrice:59,pdid:7561,stock:10000,ext:cross","name:Search,word:PET杯新品首发！,linkShadingWord:[object Object],isTiming:no"


In [6]:
import re

all_user_sku_click_explored = []
pattern = re.compile(
    r"idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)"
)

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    bid = _dict["bid"]
    search_query = _dict["search_query"]
    if not search_query or f"{search_query}" == "":
        search_query = _dict["linkInfo"]
        # 搜索pdName，如果没找到，则search_query为空字符串
        match = re.search(r"pdName:([^,]+)", search_query)
        search_query = match.group(1) if match else ""
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)

        sku_info["search_query"] = search_query
        sku_info["bid"] = bid_item
        if "undefined" in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None

                idx_match = re.search(r"idx:(\d+)", bid_item)
                if idx_match:
                    idx = idx_match.group(1)

                pdid_match = re.search(r"pdid:(\d+)", bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)

                sku_match = re.search(r"sku:([\dA-Za-z]+)", bid_item)
                if sku_match:
                    sku = sku_match.group(1)

                pid_match = re.search(r"pid:([^,]+)", bid_item)
                if pid_match:
                    pid = pid_match.group(1)

                name_match = re.search(r"name:([^,]+)", bid_item)
                if name_match:
                    name = name_match.group(1)

                sku_info.update(
                    {"idx": idx, "pdid": pdid, "sku": sku, "pid": pid, "name": name}
                )
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[
    ["pageName", "bid", "sku", "name", "idx", "pid", "pdid", "linkInfo", "search_query"]
].head(5)

,pageName,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,/search/goods,undefined,N001S01R005,阶梯价步进器,null,加购弹窗,null,"name:searchGoods,pdName:安佳淡奶油,type:2",安佳淡奶油
1,/search/goods,"name:安佳淡奶油 1L*12盒,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:337",N001S01R005,安佳淡奶油 1L*12盒,None,加购弹窗,56,"name:searchGoods,pdName:安佳淡奶油,type:2",安佳淡奶油
2,/search/goods-new,"name:台创蓝黛可可粉（低脂10%-12%）,pid:加购弹窗,sku:642483031185,pdid:7826,stock:0",642483031185,台创蓝黛可可粉（低脂10%-12%）,None,加购弹窗,7826,"name:Search,word:PET杯新品首发！,linkShadingWord:[object Object],isTiming:no",可可粉
3,/search/goods,undefined,661210374866,步进器,null,加购弹窗,null,"name:searchGoods,pdName:C味三色大芋圆,type:2",C味三色大芋圆
4,/search/goods,undefined,661210374866,步进器,null,加购弹窗,null,"name:searchGoods,pdName:C味三色大芋圆,type:2",C味三色大芋圆


In [7]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api_list', 'page_ame', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['pageName', 'idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['pageName', 'idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [8]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
4,加购弹窗,54992
5,唤起购买,43476
0,goods,33983
7,横版筛选栏,635
1,mini榜单,324
8,竖版筛选栏,51
6,换个词搜搜,16
3,个人中心,2
2,null,1


In [9]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [10]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,100027,V2,20250403,中频搜索词,0,1,1,1,1,2.0,2,2,1,1
1,100041,V2,20250403,高频搜索词,1,0,0,1,1,0.0,0,0,1,1
2,100041,V3,20250405,高频搜索词,1,0,0,1,1,0.0,0,0,1,1
3,10013,V1,20250404,高频搜索词,0,1,1,1,1,4.0,4,4,1,1
4,100144,V1,20250403,中频搜索词,1,3,4,5,5,2.4,3,2,2,1


In [11]:
null_search_query_df = pandasql.sqldf(
    """select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                    count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [12]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)


In [13]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)
# 定义计数列名列表
count_columns = ["商品详情cnt", "加入购物车cnt", "唤起购买cnt", "首屏总点击cnt", "总点击cnt"]
# 遍历计数列
for col in count_columns:
    # 将空值填充为0并转换为整数类型
    all_data_df[col] = all_data_df[col].fillna(0).astype(int)

# 创建 "用户是否点击" 列
all_data_df["用户是否点击"] = (all_data_df["总点击cnt"] > 0).astype(int)

# 定义费率计算相关列名列表
rate_columns = [
    ("sku_click_rate", "商品详情cnt", "商品查看cnt"), # 商品详情点击率
    ("add_cart_rate", "加入购物车cnt", "商品查看cnt"), # 加入购物车率
    ("popup_click_rate", "唤起购买cnt", "商品查看cnt"), # 唤起购买率
]

# 遍历费率列
for rate_col, num_col, den_col in rate_columns:
    # 计算费率，空值填充0，保留5位小数，转换为浮点数
    all_data_df[rate_col] = (all_data_df[num_col] / all_data_df[den_col]).fillna(0).round(5).astype(float)

In [14]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [15]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        # print(
        #     f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        # )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            # print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [16]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "唤起购买cnt",
    "总点击cnt",
    "商品详情cnt",
    "加入购物车cnt",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            # print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB新UI202504--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


title_all = f"搜索AB新UI202504--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
all_p_values_df

写入HTML成功！./data/搜索AB新UI202504--sku_click_rate_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--avg点击位置_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--唤起购买cnt_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--总点击cnt_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--商品详情cnt_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--加入购物车cnt_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--商品查看cnt_p-value分布-0403~0406.html
写入HTML成功！./data/搜索AB新UI202504--指标全集_p-value分布-0403~0406.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric,搜索频次
0,V1,0.1689,0.0464,0.0919,5.79,0.0,0.062500,0.166670,0.250000,0.250,0.333330,0.500000,2.00000,59,1280,497,0403~0406,sku_click_rate,中频搜索词
1,V2,1.0000,0.0439,0.0831,0.00,0.0,0.055560,0.153850,0.250000,0.250,0.330622,0.375000,1.00000,43,978,372,0403~0406,sku_click_rate,中频搜索词
2,V3,0.6510,0.0453,0.0875,3.17,0.0,0.059520,0.166670,0.250000,0.250,0.333330,0.404000,0.66667,11,248,87,0403~0406,sku_click_rate,中频搜索词
3,V4,0.8479,0.0433,0.0952,-1.42,0.0,0.050000,0.142860,0.250000,0.250,0.363332,0.500000,1.00000,11,257,88,0403~0406,sku_click_rate,中频搜索词
4,V1,0.8219,0.0394,0.0803,1.47,0.0,0.050000,0.125000,0.250000,0.250,0.250000,0.333330,1.00000,23,574,202,0403~0406,sku_click_rate,低频搜索词
5,V2,1.0000,0.0388,0.0784,0.00,0.0,0.047620,0.138506,0.241670,0.250,0.250000,0.351248,1.00000,17,429,146,0403~0406,sku_click_rate,低频搜索词
6,V3,0.7829,0.0402,0.0912,3.49,0.0,0.025640,0.166670,0.250000,0.250,0.398667,0.499000,0.75000,4,101,29,0403~0406,sku_click_rate,低频搜索词
7,V4,0.3402,0.0436,0.0947,12.38,0.0,0.040870,0.152751,0.250000,0.250,0.500000,0.500000,0.66667,4,103,30,0403~0406,sku_click_rate,低频搜索词
8,V1,0.5645,0.0424,0.0760,1.49,0.0,0.058263,0.128920,0.222220,0.250,0.285710,0.333330,1.00000,120,2826,1266,0403~0406,sku_click_rate,高频搜索词
9,V2,1.0000,0.0418,0.0760,0.00,0.0,0.055560,0.125000,0.200000,0.250,0.333330,0.375000,1.25000,92,2201,980,0403~0406,sku_click_rate,高频搜索词


In [17]:
# 导入odps_client库中的两个函数：get_odps_sql_result_as_df 用于执行SQL查询并将结果作为DataFrame返回, write_pandas_df_into_odps 用于将pandas DataFrame写入ODPS表
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

# 定义分区规范字符串，使用当前日期（年-月-日格式）作为分区值
partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

# 将DataFrame写入ODPS表
write_pandas_df_into_odps(
    df=all_user_variant_df,  # 要写入的DataFrame，这里是all_user_variant_df，包含了所有用户的变体信息
    table_name="summerfarm_ds.temp_search_ab_all_data_df",  # ODPS表名
    partition_spec=partition_spec,  # 分区规范
    overwrite=True,  # 如果表或分区已存在，是否覆盖
    lifecycle=30,  # 设置表的生命周期为30天
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

# 将开始日期格式化为字符串（年-月-日）
start_date_str = start_date.strftime("%Y-%m-%d")

# 定义SQL查询字符串，用于获取用户订单数据.
# 这段SQL的目的是：从订单表和用户分流表中，根据用户ID和日期进行关联，
# 统计每个用户在不同实验变体下的订单总金额、订单数量和平均订单金额。
order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

# 执行SQL查询并将结果作为DataFrame返回
user_orders_df = get_odps_sql_result_as_df(order_query)
# 显示DataFrame的前两行
user_orders_df.head(2)

2025-04-07 18:42:18 - INFO - DaraFrame字段合集:api,experiment_id,variant_list,page_ame,type,uid,ds,search_times,api_list,create_time,pt
2025-04-07 18:42:22 - INFO - Tunnel session created: <TableUploadSession id=20250407184222f001c30b0dcb5763 project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20250407>
2025-04-07 18:42:26 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20250407, attemp:0
2025-04-07 18:42:35 - INFO - Tunnel session created: <InstanceDownloadSession id=20250407184234f001c30b0dcb5ce0 project_name=summerfarm_ds instance_id=20250407104226734gbp9l7mzi22>
2025-04-07 18:42:36 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-04-03 00:00:00'
    AND     m_size = '单店'
),user

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250403,494720,V2,807,2,403.5
1,20250403,501169,"V1,V2",1835,3,611.67


In [18]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count     16061.000000
mean        587.853873
std        1247.366746
min           0.380000
25%         180.000000
50%         333.000000
75%         641.500000
max      107136.000000
Name: order_gmv, dtype: float64

In [19]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500       333.0000
0.750       641.5000
0.950      1718.0000
0.990      3965.0000
0.995      5554.8000
0.996      5986.8800
0.997      6720.8406
0.999     11895.1800
1.000    107136.0000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      48.9996
0.05      83.7200
0.25     179.5000
0.50     331.0000
0.75     636.0000
0.95    1656.0000
0.99    3217.8200
Name: order_gmv, dtype: float64


In [20]:
user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(
    float
)

user_orders_below_6k_df["avg_order_gmv"].fillna(0.0, inplace=True)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df[
    "avg_order_gmv"
].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB新UI202504--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_21929/2812266756.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_21929/2812266756.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["ord

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.8052,537.8341,617.0878,-0.50,333.0,635.00,1120.72,1640.0,2033.92,3165.52,4301.68,5990.0,953714,1773,1773,0403~0406,order_gmv
1,V2,1.0000,540.5456,623.1145,0.00,325.2,633.00,1133.04,1640.0,2078.56,3215.77,4118.52,5903.0,781494,1446,1446,0403~0406,order_gmv
2,V3,0.9071,542.7704,659.9740,0.41,325.5,629.00,1120.00,1621.0,2102.64,3389.08,4848.60,5977.0,199332,367,367,0403~0406,order_gmv
3,V4,0.3173,559.5680,643.7757,3.52,335.0,642.25,1216.00,1726.0,2214.50,3127.48,3887.51,5955.0,196828,352,352,0403~0406,order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.9525,439.2351,479.3310,0.12,279.0,544.80,940.0,1256.20,1630.00,2411.240,2957.35,5990.0,778874,1773,1773,0403~0406,avg_order_gmv
1,V2,1.0000,438.7265,484.2626,0.00,277.0,540.75,940.0,1265.25,1592.16,2441.620,3010.72,5900.0,634289,1446,1446,0403~0406,avg_order_gmv
2,V3,0.9826,438.4058,507.8331,-0.07,279.0,535.00,932.4,1195.60,1599.20,2291.640,3496.40,5800.0,161005,367,367,0403~0406,avg_order_gmv
3,V4,0.5319,447.7377,484.9806,2.05,276.0,551.00,940.0,1370.00,1623.97,2411.525,3050.00,4660.0,157492,352,352,0403~0406,avg_order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.0891,1.2566,0.6086,-1.56,1.0,1.0,2.0,2.0,3.0,4.0,4.0,8,2228,1773,1773,0403~0406,order_cnt
1,V2,1.0000,1.2765,0.7007,0.00,1.0,1.0,2.0,2.0,3.0,4.0,5.0,10,1846,1446,1446,0403~0406,order_cnt
2,V3,0.8851,1.2737,0.6664,-0.22,1.0,1.0,2.0,3.0,3.0,4.0,5.0,8,468,367,367,0403~0406,order_cnt
3,V4,0.7458,1.2836,0.7428,0.55,1.0,1.0,2.0,3.0,3.0,4.0,5.0,14,452,352,352,0403~0406,order_cnt


写入HTML成功！./data/搜索AB新UI202504--订单转化p-value分布-0403~0406.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.8052,537.8341,617.0878,-0.50,333.0,635.00,1120.72,1640.00,2033.92,3165.520,4301.68,5990.0,953714,1773,1773,0403~0406,order_gmv
1,V2,1.0000,540.5456,623.1145,0.00,325.2,633.00,1133.04,1640.00,2078.56,3215.770,4118.52,5903.0,781494,1446,1446,0403~0406,order_gmv
2,V3,0.9071,542.7704,659.9740,0.41,325.5,629.00,1120.00,1621.00,2102.64,3389.080,4848.60,5977.0,199332,367,367,0403~0406,order_gmv
3,V4,0.3173,559.5680,643.7757,3.52,335.0,642.25,1216.00,1726.00,2214.50,3127.480,3887.51,5955.0,196828,352,352,0403~0406,order_gmv
4,V1,0.9525,439.2351,479.3310,0.12,279.0,544.80,940.00,1256.20,1630.00,2411.240,2957.35,5990.0,778874,1773,1773,0403~0406,avg_order_gmv
5,V2,1.0000,438.7265,484.2626,0.00,277.0,540.75,940.00,1265.25,1592.16,2441.620,3010.72,5900.0,634289,1446,1446,0403~0406,avg_order_gmv
6,V3,0.9826,438.4058,507.8331,-0.07,279.0,535.00,932.40,1195.60,1599.20,2291.640,3496.40,5800.0,161005,367,367,0403~0406,avg_order_gmv
7,V4,0.5319,447.7377,484.9806,2.05,276.0,551.00,940.00,1370.00,1623.97,2411.525,3050.00,4660.0,157492,352,352,0403~0406,avg_order_gmv
8,V1,0.0891,1.2566,0.6086,-1.56,1.0,1.00,2.00,2.00,3.00,4.000,4.00,8.0,2228,1773,1773,0403~0406,order_cnt
9,V2,1.0000,1.2765,0.7007,0.00,1.0,1.00,2.00,2.00,3.00,4.000,5.00,10.0,1846,1446,1446,0403~0406,order_cnt


In [21]:
all_p_values_df.to_csv(
    f"./data/搜索AB新UI202504--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB新UI202504--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)